# A/B Test 02 — Conversion Rate

**Question.** Did the `treatment` group purchase at a higher rate than `control`, and should we be confident the effect is real?

**Metric type.** `purchased` is binary 0/1 → conversion rate is a **proportion** → **two-proportion z-test**.


In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet("../data/ab_test_data.parquet")
df.head()

,user_id,group,clicked,purchased,order_value,returned_30d
0,T01163,treatment,1,0,0.00,1
1,T04385,treatment,0,0,0.00,1
2,C01902,control,1,1,96.31,0
3,C03397,control,0,0,0.00,0
4,T05695,treatment,1,0,0.00,1


In [2]:
# Data quality checks (run before trusting any metric)
print("shape:", df.shape)
print("\nnulls:\n", df.isnull().sum())
print("\nduplicate user_id:", df["user_id"].duplicated().sum())

# SRM (sample ratio mismatch): groups should be ~50/50
print("\ngroup sizes:\n", df["group"].value_counts())

# Binary columns must contain only 0/1
for col in ["clicked", "purchased", "returned_30d"]:
    print(col, "unique:", sorted(df[col].unique()))

shape: (12000, 6)

nulls:
 user_id         0
group           0
clicked         0
purchased       0
order_value     0
returned_30d    0
dtype: int64

duplicate user_id: 0

group sizes:
 group
treatment    6000
control      6000
Name: count, dtype: int64
clicked unique: [np.int64(0), np.int64(1)]
purchased unique: [np.int64(0), np.int64(1)]
returned_30d unique: [np.int64(0), np.int64(1)]


## Compute the metric

In [3]:
summary = df.groupby("group")["purchased"].agg(["sum", "count"])
summary["conversion_rate"] = summary["sum"] / summary["count"]
print(summary)

           sum  count  conversion_rate
group                                 
control    327   6000         0.054500
treatment  551   6000         0.091833


## Statistical test — two-proportion z-test

In [4]:
conversions = summary["sum"].values
n           = summary["count"].values
stat, pval = proportions_ztest(conversions, n)

print(f"control = {summary['conversion_rate']['control']:.4f} | treatment = {summary['conversion_rate']['treatment']:.4f}")
print(f"p-value = {pval:.4f}")
print("Significant" if pval < 0.05 else "Not significant")

control = 0.0545 | treatment = 0.0918
p-value = 0.0000
Significant


## Result

Conversion rose from **5.45% (control)** to **9.18% (treatment)**, p ≈ 0.0000 → significant.

- **Absolute lift:** +3.7 percentage points
- **Relative lift:** ≈ +68%


## Concepts

### Conversion measures buyers, not money
Conversion only tells you **how many** people bought, not **how much** they spent.

> Revenue = number of buyers × spend per buyer

It is possible for conversion to rise while total revenue falls (e.g. more people buy, but each buys cheaper items). Conversion alone cannot tell these apart — **revenue / ARPU** can.

### The reusable "should we ship?" pattern
Significance on the primary metric is necessary but not sufficient. Before shipping:

1. **Primary metric** significant and in the right direction?
2. **Downstream** metrics improved? (conversion → revenue / ARPU)
3. **Guardrail** metrics not worse? (returns, complaints, page speed)
4. Did the test run **long enough** to rule out a novelty effect, and is the lift **practically** large enough?
